In [7]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))
from src import constants as cts
from src import helper as hp
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from google import genai
from src import reviewer
from src.llm import LLMProvider

In [2]:
det_path = list(Path("detections").glob("*.csv"))

print(det_path)
dets = pd.read_csv(det_path[0])

[WindowsPath('detections/catalog.csv')]


In [3]:
dets = dets[dets["z_shift"] > 0].sort_values(by="z_shift", ascending=False)
#det = pd.to_datetime(dets.iloc[0]["Date and time"])
dets[cts.DET_TS_COL] = pd.to_datetime(dets[cts.DET_TS_COL])

det = dets[(dets[cts.DET_TS_COL] == pd.to_datetime("2021-01-21 12:30:00"))]
dets.head()




,WT_ID,re_at_ts,Date and time,signal_name,window_start,window_end,wind_center,n_baseline,mean_baseline,std_baseline,mean_event,std_event,value_at_ts,delta_at_ts,z_at_ts,delta_mean,z_shift
69,2,0.684403,2021-01-21 12:30:00,Generator bearing front temperature (°C),2021-01-11 02:10:00,2021-01-26 15:10:00,19.5,231,40.436923,4.409229,60.145190,10.842733,66.108621,25.671698,5.822265,19.708268,4.469777
738,15,1.219538,2022-09-04 04:10:00,Rotor bearing temp (°C),2022-08-30 04:30:00,2022-09-09 04:30:00,10.5,1531,28.206051,2.335001,36.070456,0.419233,35.471666,7.265615,3.111611,7.864405,3.368052
690,14,1.151810,2021-06-25 09:00:00,Generator bearing front temperature (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,37.357630,3.338454,44.639760,2.647685,43.001666,5.644036,1.690613,7.282130,2.181288
688,14,1.195350,2021-06-25 09:00:00,Rotor bearing temp (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,26.837207,2.484278,32.101889,0.462255,31.605000,4.767793,1.919187,5.264682,2.119200
550,12,0.907640,2020-10-10 19:00:00,Generator bearing front temperature (°C),2020-10-05 15:50:00,2020-10-15 21:10:00,11.5,1155,37.900581,4.653485,47.562265,11.513257,68.943334,31.042753,6.670862,9.661683,2.076225


In [4]:
dets.head()

,WT_ID,re_at_ts,Date and time,signal_name,window_start,window_end,wind_center,n_baseline,mean_baseline,std_baseline,mean_event,std_event,value_at_ts,delta_at_ts,z_at_ts,delta_mean,z_shift
69,2,0.684403,2021-01-21 12:30:00,Generator bearing front temperature (°C),2021-01-11 02:10:00,2021-01-26 15:10:00,19.5,231,40.436923,4.409229,60.145190,10.842733,66.108621,25.671698,5.822265,19.708268,4.469777
738,15,1.219538,2022-09-04 04:10:00,Rotor bearing temp (°C),2022-08-30 04:30:00,2022-09-09 04:30:00,10.5,1531,28.206051,2.335001,36.070456,0.419233,35.471666,7.265615,3.111611,7.864405,3.368052
690,14,1.151810,2021-06-25 09:00:00,Generator bearing front temperature (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,37.357630,3.338454,44.639760,2.647685,43.001666,5.644036,1.690613,7.282130,2.181288
688,14,1.195350,2021-06-25 09:00:00,Rotor bearing temp (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,26.837207,2.484278,32.101889,0.462255,31.605000,4.767793,1.919187,5.264682,2.119200
550,12,0.907640,2020-10-10 19:00:00,Generator bearing front temperature (°C),2020-10-05 15:50:00,2020-10-15 21:10:00,11.5,1155,37.900581,4.653485,47.562265,11.513257,68.943334,31.042753,6.670862,9.661683,2.076225


In [5]:
log_path = list(Path("status_logs").glob("*.csv"))

print(log_path)
log1 = pd.read_csv(log_path[0])
log1.head()

[WindowsPath('status_logs/Status_Logs_Complete_ID_1.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_10.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_11.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_12.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_13.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_14.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_15.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_2.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_4.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_5.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_6.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_7.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_8.csv'), WindowsPath('status_logs/Status_Logs_Complete_ID_9.csv')]


C:\Users\admin\AppData\Local\Temp\ipykernel_2296\495428329.py:4: DtypeWarning: Columns (0: Comment) have mixed types. Specify dtype option on import or set low_memory=False.
  log1 = pd.read_csv(log_path[0])


,Timestamp start,Timestamp end,Duration,Status,Code,Message,Comment,Service contract category,IEC category,WT_ID,Global contract category,Custom contract category
0,2016-06-06 17:08:40,2016-06-10 11:28:00,90:19:20,Stop,3500,Grid loss,NaN,External stop (grid) (4),Out of Electrical Specification,1,NaN,NaN
1,2016-06-06 17:08:41,2016-06-06 18:14:07,01:05:26,Warning,3875,Overload transf. fan inlet air,NaN,Warnings (27),NaN,1,NaN,NaN
2,2016-06-06 17:08:41,2016-06-06 18:14:39,01:05:58,Warning,1825,Overload gear bypass filter,NaN,Warnings (27),NaN,1,NaN,NaN
3,2016-06-06 17:08:41,2016-06-06 18:14:07,01:05:26,Warning,3875,Overload transf. fan inlet air,NaN,Warnings (27),NaN,1,NaN,NaN
4,2016-06-06 17:08:41,2016-06-06 18:14:39,01:05:58,Warning,1825,Overload gear bypass filter,NaN,Warnings (27),NaN,1,NaN,NaN


In [6]:

chunks = hp.load_logfile(list(Path(cts.PATH_STATUS_LOGS).glob("*.csv"))[0],
                                 wt_col=None,
                                 wt_id=None,
                                 detection_ts=pd.to_datetime("2020-11-22 20:00:00"),
                                 ts_cols=["Timestamp start", "Timestamp end"], 
                                 delta_time=pd.Timedelta(days=30)
                                 )

print(len(chunks))
print(chunks)
chunks[0].empty

14
[          Timestamp start       Timestamp end  Duration         Status  Code  \
43648 2020-10-26 06:47:38 2020-10-26 06:49:56  00:02:18           Stop   710   
43660 2020-10-28 12:23:47 2020-10-28 13:50:17  01:26:30           Stop    25   
43662 2020-10-28 12:24:34 2020-10-28 13:50:17  01:25:43  Informational  6410   
43663 2020-10-28 13:25:20 2020-10-28 13:50:23  00:25:03        Warning  2125   
43671 2020-10-29 07:44:50 2020-10-29 07:45:38  00:00:48  Informational    10   
43679 2020-10-30 20:15:30 2020-10-30 20:16:18  00:00:48  Informational    10   
43687 2020-10-30 20:46:12 2020-10-30 20:47:00  00:00:48  Informational    10   
43695 2020-10-30 21:36:32 2020-10-30 21:37:20  00:00:48  Informational    10   
43703 2020-11-01 21:04:17 2020-11-01 21:05:05  00:00:48  Informational    10   
43711 2020-11-02 07:01:09 2020-11-02 07:03:27  00:02:18           Stop   710   
43723 2020-11-02 07:42:05 2020-11-02 07:42:52  00:00:47  Informational    10   
43731 2020-11-02 08:01:51 2020-11-02

False

In [7]:
from src import constants as cts
print(list(Path(cts.PATH_STATUS_LOGS).glob("*.csv"))[0])



C:\Users\admin\Programming\llms\llm-wind-turbine-status-log-review\examples\status_logs\Status_Logs_Complete_ID_1.csv


In [8]:
from src import prompt as p

format = p.format_chunk_logs(chunk_log=chunks[0], columns=cts.LOG_COLS)

print(format)

df_index=43648;Timestamp start=2020-10-26 06:47:38;Timestamp end=2020-10-26 06:49:56;WT_ID=1;Status=Stop;Code=710;Message=Battery test;Comment=<NA>
df_index=43660;Timestamp start=2020-10-28 12:23:47;Timestamp end=2020-10-28 13:50:17;WT_ID=1;Status=Stop;Code=25;Message=Manual stop without login;Comment=<NA>
df_index=43662;Timestamp start=2020-10-28 12:24:34;Timestamp end=2020-10-28 13:50:17;WT_ID=1;Status=Informational;Code=6410;Message=Manual yaw;Comment=<NA>
df_index=43663;Timestamp start=2020-10-28 13:25:20;Timestamp end=2020-10-28 13:50:23;WT_ID=1;Status=Warning;Code=2125;Message=Timeout brake closed;Comment=<NA>
df_index=43671;Timestamp start=2020-10-29 07:44:50;Timestamp end=2020-10-29 07:45:38;WT_ID=1;Status=Informational;Code=10;Message=Wind < start wind;Comment=<NA>
df_index=43679;Timestamp start=2020-10-30 20:15:30;Timestamp end=2020-10-30 20:16:18;WT_ID=1;Status=Informational;Code=10;Message=Wind < start wind;Comment=<NA>
df_index=43687;Timestamp start=2020-10-30 20:46:12;Tim

In [9]:
# from src import llm

# raw_res = llm.call_ollama(
#     prompt=prompt,
#     model="llama3.2:3b",
#     temperature=0.0,
# )

# print(raw_res)

In [10]:
dets.head()

,WT_ID,re_at_ts,Date and time,signal_name,window_start,window_end,wind_center,n_baseline,mean_baseline,std_baseline,mean_event,std_event,value_at_ts,delta_at_ts,z_at_ts,delta_mean,z_shift
69,2,0.684403,2021-01-21 12:30:00,Generator bearing front temperature (°C),2021-01-11 02:10:00,2021-01-26 15:10:00,19.5,231,40.436923,4.409229,60.145190,10.842733,66.108621,25.671698,5.822265,19.708268,4.469777
738,15,1.219538,2022-09-04 04:10:00,Rotor bearing temp (°C),2022-08-30 04:30:00,2022-09-09 04:30:00,10.5,1531,28.206051,2.335001,36.070456,0.419233,35.471666,7.265615,3.111611,7.864405,3.368052
690,14,1.151810,2021-06-25 09:00:00,Generator bearing front temperature (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,37.357630,3.338454,44.639760,2.647685,43.001666,5.644036,1.690613,7.282130,2.181288
688,14,1.195350,2021-06-25 09:00:00,Rotor bearing temp (°C),2021-06-20 06:30:00,2021-06-30 09:50:00,15.5,657,26.837207,2.484278,32.101889,0.462255,31.605000,4.767793,1.919187,5.264682,2.119200
550,12,0.907640,2020-10-10 19:00:00,Generator bearing front temperature (°C),2020-10-05 15:50:00,2020-10-15 21:10:00,11.5,1155,37.900581,4.653485,47.562265,11.513257,68.943334,31.042753,6.670862,9.661683,2.076225


In [ ]:


# wt_id = dets.iloc[0]["WT_ID"]

# log = [p for p in log_path if f"_ID_{wt_id}." in p.name]

# answer = reviewer.review_detection(detection=dets.iloc[0],
#                           log_file_path=log[0],
#                           delta_time=pd.Timedelta(days=30),
#                           ts_cols=cts.TS_COLS,
#                           detection_ts_col=cts.DET_TS_COL,
#                           wt_id_col=cts.WT_ID,
#                           signal_col=cts.SIGNAL,
#                           provider=LLMProvider.GEMINI,
#                           model="gemini-2.5-flash-lite")

# print(answer.model_dump())

{'wt_id': '2', 'detection_id': '69', 'detections_ts': '2021-01-21 12:30:00', 'relevant_signal': 'Generator bearing front temperature (°C)', 'anomaly_description_reasoning': 'The anomaly detection indicates a significant increase in the Generator bearing front temperature (°C) at 2021-01-21 12:30:00. The statistics show a mean event temperature of 60.15°C compared to a baseline mean of 40.44°C, with the value at the detection timestamp being 66.11°C. This suggests a potential overheating issue with the front generator bearing.', 'relevant_logs': [], 'overall_assessment': 'possibly undocumented anomaly', 'overall_reasoning': "The anomaly statistics clearly indicate a significant deviation in the Generator bearing front temperature. However, none of the provided status logs offer concrete evidence directly related to this specific temperature anomaly or its potential causes or immediate precursors. The logs are either too old, too generic (e.g., 'Wind < start wind'), or related to unrelat

In [ ]:
# print(answer.model_dump_json(indent=2))

{
  "wt_id": "2",
  "detection_id": "69",
  "detections_ts": "2021-01-21 12:30:00",
  "relevant_signal": "Generator bearing front temperature (°C)",
  "anomaly_description_reasoning": "The anomaly detection indicates a significant increase in the Generator bearing front temperature (°C) at 2021-01-21 12:30:00. The statistics show a mean event temperature of 60.15°C compared to a baseline mean of 40.44°C, with the value at the detection timestamp being 66.11°C. This suggests a potential overheating issue with the front generator bearing.",
  "relevant_logs": [],
  "overall_assessment": "possibly undocumented anomaly",
  "overall_reasoning": "The anomaly statistics clearly indicate a significant deviation in the Generator bearing front temperature. However, none of the provided status logs offer concrete evidence directly related to this specific temperature anomaly or its potential causes or immediate precursors. The logs are either too old, too generic (e.g., 'Wind < start wind'), or r

In [13]:
# client = genai.Client()

# for model in client.models.list():
#     print(model.name)
#     print("supported actions:", model.supported_actions)
#     print()

In [8]:
outputs = reviewer.deep_search(
    detections=dets,
    log_path=log_path,
    provider=LLMProvider.GEMINI,
    model="gemini-2.5-flash-lite")





c:\Users\admin\Programming\llms\llm-wind-turbine-status-log-review\src\parser.py:10: UserWarning: LLM output format is faulty
  warnings.warn(f"LLM output format is faulty")


ValueError: Error: 4 validation errors for LLMOutput
relevant_logs.0.relevance
  Extra inputs are not permitted [type=extra_forbidden, input_value='medium', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
relevant_logs.0.influence
  Extra inputs are not permitted [type=extra_forbidden, input_value='indirect', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
relevant_logs.1.relevance
  Extra inputs are not permitted [type=extra_forbidden, input_value='weak', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
relevant_logs.1.influence
  Extra inputs are not permitted [type=extra_forbidden, input_value='indirect', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden

In [ ]:
result = pd.DataFrame([output.model_dump for output in outputs if not output.relevant_logs])

result.head()